# Bulk File Converter

Converts `.html` and `.json` files from a folder into your preferred output format.

**Supported output formats:** `.md` · `.txt` · `.docx`

---

## Steps
1. Run **Cell 1** to install dependencies (once)
2. Set your preferences in **Cell 2 (Configuration)**
3. Run all remaining cells to convert your files

In [ ]:
# Cell 1 — Install dependencies (run once)
%pip install html2text beautifulsoup4 python-docx lxml --quiet

In [ ]:
# Cell 2 — CONFIGURATION  ← edit these values before running

# Folder containing your .html and/or .json files
INPUT_FOLDER = "/path/to/your/input/folder"

# Where converted files will be saved (will be created if it doesn't exist)
# Set to None to save alongside the originals
OUTPUT_FOLDER = "/path/to/your/output/folder"

# Output format — choose ONE: "md", "txt", or "docx"
OUTPUT_FORMAT = "md"

# Which file types to convert — comment out any you don't want
CONVERT_HTML = True
CONVERT_JSON = True

# If True, existing output files will be overwritten
OVERWRITE = False

In [ ]:
# Cell 3 — Imports
import json
import os
from pathlib import Path

import html2text
from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

print("Imports OK")

In [ ]:
# Cell 4 — HTML conversion helpers

def _html_to_plain(html_content: str) -> str:
    """Strip all tags; return plain text."""
    soup = BeautifulSoup(html_content, "lxml")
    return soup.get_text(separator="\n", strip=True)


def _html_to_markdown(html_content: str) -> str:
    converter = html2text.HTML2Text()
    converter.ignore_links = False
    converter.ignore_images = False
    converter.body_width = 0          # no hard line wrapping
    converter.protect_links = True
    return converter.handle(html_content)


def _html_to_docx(html_content: str) -> Document:
    doc = Document()
    soup = BeautifulSoup(html_content, "lxml")

    heading_map = {"h1": 1, "h2": 2, "h3": 3, "h4": 4, "h5": 5, "h6": 6}

    for element in soup.body.descendants if soup.body else soup.descendants:
        if not hasattr(element, "name") or element.name is None:
            continue
        name = element.name.lower()

        if name in heading_map:
            doc.add_heading(element.get_text(strip=True), level=heading_map[name])

        elif name == "p":
            text = element.get_text(strip=True)
            if text:
                doc.add_paragraph(text)

        elif name in ("ul", "ol"):
            for li in element.find_all("li", recursive=False):
                doc.add_paragraph(li.get_text(strip=True), style="List Bullet")

        elif name in ("pre", "code"):
            para = doc.add_paragraph(element.get_text())
            run = para.runs[0] if para.runs else para.add_run()
            run.font.name = "Courier New"
            run.font.size = Pt(10)

        elif name == "blockquote":
            para = doc.add_paragraph(element.get_text(strip=True))
            para.paragraph_format.left_indent = Pt(24)

        elif name == "hr":
            doc.add_paragraph("─" * 60)

    return doc


print("HTML helpers defined")

In [ ]:
# Cell 5 — JSON conversion helpers

def _json_to_markdown(data, level: int = 1, _root: bool = True) -> str:
    """Recursively render a JSON object as Markdown."""
    lines = []
    prefix = "#" * min(level, 6)

    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, (dict, list)):
                lines.append(f"{prefix} {key}")
                lines.append(_json_to_markdown(value, level + 1, _root=False))
            else:
                lines.append(f"**{key}:** {value}")

    elif isinstance(data, list):
        for i, item in enumerate(data):
            if isinstance(item, (dict, list)):
                lines.append(f"{prefix} Item {i + 1}")
                lines.append(_json_to_markdown(item, level + 1, _root=False))
            else:
                lines.append(f"- {item}")
    else:
        lines.append(str(data))

    return "\n".join(lines)


def _json_to_plain(data) -> str:
    return json.dumps(data, indent=2, ensure_ascii=False)


def _json_to_docx(data) -> Document:
    doc = Document()

    def _write(node, level: int = 1):
        if isinstance(node, dict):
            for key, value in node.items():
                if isinstance(value, (dict, list)):
                    doc.add_heading(str(key), level=min(level, 6))
                    _write(value, level + 1)
                else:
                    para = doc.add_paragraph()
                    run_key = para.add_run(f"{key}: ")
                    run_key.bold = True
                    para.add_run(str(value))

        elif isinstance(node, list):
            for i, item in enumerate(node):
                if isinstance(item, (dict, list)):
                    doc.add_heading(f"Item {i + 1}", level=min(level, 6))
                    _write(item, level + 1)
                else:
                    doc.add_paragraph(str(item), style="List Bullet")
        else:
            doc.add_paragraph(str(node))

    _write(data)
    return doc


print("JSON helpers defined")

In [ ]:
# Cell 6 — Core converter

def convert_file(src: Path, dest_dir: Path, fmt: str, overwrite: bool) -> str:
    """
    Convert a single file and write it to dest_dir.
    Returns a short status string.
    """
    fmt = fmt.lower().lstrip(".")
    assert fmt in ("md", "txt", "docx"), f"Unsupported format: {fmt}"

    dest_path = dest_dir / f"{src.stem}.{fmt}"

    if dest_path.exists() and not overwrite:
        return f"  SKIP  {src.name}  →  {dest_path.name}  (already exists)"

    ext = src.suffix.lower()
    raw = src.read_text(encoding="utf-8", errors="replace")

    # ── HTML ──────────────────────────────────────────────────────────────
    if ext == ".html":
        if fmt == "md":
            text = _html_to_markdown(raw)
            dest_path.write_text(text, encoding="utf-8")
        elif fmt == "txt":
            text = _html_to_plain(raw)
            dest_path.write_text(text, encoding="utf-8")
        elif fmt == "docx":
            doc = _html_to_docx(raw)
            doc.save(dest_path)

    # ── JSON ──────────────────────────────────────────────────────────────
    elif ext == ".json":
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            return f"  ERROR {src.name}  →  JSON parse failed: {e}"

        if fmt == "md":
            text = _json_to_markdown(data)
            dest_path.write_text(text, encoding="utf-8")
        elif fmt == "txt":
            text = _json_to_plain(data)
            dest_path.write_text(text, encoding="utf-8")
        elif fmt == "docx":
            doc = _json_to_docx(data)
            doc.save(dest_path)

    else:
        return f"  SKIP  {src.name}  →  unsupported source type"

    return f"  OK    {src.name}  →  {dest_path.name}"


print("Converter defined")

In [ ]:
# Cell 7 — Validate config before running

input_path = Path(INPUT_FOLDER)
assert input_path.exists(), f"Input folder not found: {INPUT_FOLDER}"
assert input_path.is_dir(), f"Not a directory: {INPUT_FOLDER}"
assert OUTPUT_FORMAT.lower().lstrip(".") in ("md", "txt", "docx"), \
    f"Invalid OUTPUT_FORMAT '{OUTPUT_FORMAT}' — choose 'md', 'txt', or 'docx'"

output_path = Path(OUTPUT_FOLDER) if OUTPUT_FOLDER else input_path
output_path.mkdir(parents=True, exist_ok=True)

# Collect files
candidates = []
if CONVERT_HTML:
    candidates += list(input_path.glob("*.html")) + list(input_path.glob("*.htm"))
if CONVERT_JSON:
    candidates += list(input_path.glob("*.json"))

print(f"Input  : {input_path}")
print(f"Output : {output_path}")
print(f"Format : .{OUTPUT_FORMAT.lower().lstrip('.')}")
print(f"Files  : {len(candidates)} file(s) found")
for f in candidates:
    print(f"         {f.name}")

In [ ]:
# Cell 8 — Run bulk conversion

if not candidates:
    print("No files to convert. Check INPUT_FOLDER and CONVERT_HTML / CONVERT_JSON flags.")
else:
    fmt = OUTPUT_FORMAT.lower().lstrip(".")
    results = []
    for src in sorted(candidates):
        status = convert_file(src, output_path, fmt, OVERWRITE)
        results.append(status)
        print(status)

    ok    = sum(1 for r in results if r.startswith("  OK"))
    skip  = sum(1 for r in results if r.startswith("  SKIP"))
    error = sum(1 for r in results if r.startswith("  ERROR"))

    print(f"\n{'─'*50}")
    print(f"Done — {ok} converted, {skip} skipped, {error} errors.")
    print(f"Output folder: {output_path.resolve()}")